In [1]:
import json
import logging
import random
import re

# import brotli
import requests
from playwright.async_api import async_playwright

logging.basicConfig(level=logging.NOTSET)
handle = "tesco_api"
logger = logging.getLogger(handle)

user_agent_strings = [
    "Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/45.0.2454.85 Safari/537.36",
    "Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/45.0.2454.85 Safari/537.36",
]

In [ ]:
# Get supervalu product


session = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "x-site-host": "https://shop.supervalu.ie",
    "x-site-location": "HeadersBuilderInterceptor",
    "x-shopping-mode": "11111111-1111-1111-1111-111111111111",
    "Referer": "https://shop.supervalu.ie/",
}
# First visit the main site to get cookies
# session.get("https://shop.supervalu.ie", headers=headers)

# Then try the API
# url = "https://storefrontgateway.supervalu.ie/api/stores/831/products/1562108000"

for i in [831, 250]:
    url = f"https://storefrontgateway.supervalu.ie/api/stores/{i}/products/1882507000"
    response = session.get(url, headers=headers)
    if response:
        break
    else:
        pass

print(response.status_code)
# supervalu_product_info_decompressed = brotli.decompress(response.content)
supervalu_product_info = json.loads(response.content)

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): storefrontgateway.supervalu.ie:443
DEBUG:urllib3.connectionpool:https://storefrontgateway.supervalu.ie:443 "GET /api/stores/831/products/1882507000 HTTP/1.1" 404 None
DEBUG:urllib3.connectionpool:https://storefrontgateway.supervalu.ie:443 "GET /api/stores/250/products/1882507000 HTTP/1.1" 200 None


200


In [ ]:
if response:
    print("yes")
else:
    print("No")

yes


In [30]:
supervalu_product_info

{'name': 'Fonte Real Alicante Bouschet Reserva (75 cl)',
 'defaultCategory': 'Portugal',
 'categories': [{'categoryId': '21742f38-b3ec-452c-be59-d153cdb8b74e',
   'retailerId': 'Grocery',
   'category': 'Grocery',
   'categoryBreadcrumb': '/Grocery'},
  {'categoryId': 'ae769466-d42e-46bc-9c77-5fa7e10fd8d6',
   'retailerId': 'O100075',
   'category': 'Wine, Beer & Spirits',
   'categoryBreadcrumb': '/Grocery/Wine, Beer & Spirits'},
  {'categoryId': '85a59995-dda1-46c6-b2e8-11941707c75e',
   'retailerId': 'O200590',
   'category': 'Wine',
   'categoryBreadcrumb': '/Grocery/Wine, Beer & Spirits/Wine'},
  {'categoryId': 'e99aef46-d754-4ef1-93df-58ec26fe3ac6',
   'retailerId': 'O302535',
   'category': 'Red Wine',
   'categoryBreadcrumb': '/Grocery/Wine, Beer & Spirits/Wine/Red Wine'},
  {'categoryId': 'ed5607ce-463e-43b5-80c5-e4ed7b713abb',
   'retailerId': 'O410006',
   'category': 'Portugal',
   'categoryBreadcrumb': '/Grocery/Wine, Beer & Spirits/Wine/Red Wine/Portugal'}],
 'sku': '1882

In [ ]:
# Get Supervalu sitemap XML to extract all UK product ids
async def fetch_sitemap():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=False, args=["--disable-blink-features=AutomationControlled"]
        )

        context = await browser.new_context(
            user_agent=random.choice(user_agent_strings),
            viewport={"width": 1920, "height": 1080},
        )

        await context.add_init_script(
            "Object.defineProperty(navigator, 'webdriver', { get: () => undefined })"
        )
        page = await browser.new_page()
        await page.goto("https://shop.supervalu.ie/sitemap.xml", wait_until="domcontentloaded")
        await page.wait_for_timeout(5000)
        content = await page.content()
        await browser.close()
        return content


supervalu_xml = await fetch_sitemap()
supervalu_products = re.findall(
    r"<loc>(https://shop.supervalu.ie/product/[^<]+)</loc>", supervalu_xml
)
supervalu_ids = re.findall(r"-(\d+)</loc>", supervalu_xml)

In [ ]:
[url for url in supervalu_products if "1882507000" in url]

['https://shop.supervalu.ie/product/fonte-real-alicante-bouschet-reserva-75-cl-id-1882507000']

In [30]:
supervalu_ids = re.findall(r"-(\d+)</loc>", supervalu_xml)
len(set(supervalu_ids))

11807

In [ ]:
async def intercept_product_api():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context()
        page = await context.new_page()

        # Capture all requests
        requests_log = []

        def log_request(request):
            if "product" in request.url.lower() or "storefrontgateway" in request.url:
                requests_log.append(
                    {"url": request.url, "method": request.method, "headers": dict(request.headers)}
                )

        page.on("request", log_request)

        await page.goto(
            "https://shop.supervalu.ie/sm/pickup/rsid/250/product/fonte-real-alicante-bouschet-reserva-75-cl-id-1882507000"
        )
        await page.wait_for_timeout(5000)

        await browser.close()

        for req in requests_log:
            print(f"\n{req['method']} {req['url']}")
            print(f"Headers: {json.dumps(req['headers'], indent=2)}")

        return requests_log


requests = await intercept_product_api()


GET https://shop.supervalu.ie/sm/pickup/rsid/250/product/fonte-real-alicante-bouschet-reserva-75-cl-id-1882507000
Headers: {
  "upgrade-insecure-requests": "1",
  "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
  "sec-ch-ua": "\"Chromium\";v=\"143\", \"Not A(Brand\";v=\"24\"",
  "sec-ch-ua-mobile": "?0",
  "sec-ch-ua-platform": "\"macOS\""
}

GET https://shop.supervalu.ie/static/js/ProductDetails.5ce8aab1.chunk.js
Headers: {
  "sec-ch-ua-platform": "\"macOS\"",
  "referer": "https://shop.supervalu.ie/sm/pickup/rsid/250/product/fonte-real-alicante-bouschet-reserva-75-cl-id-1882507000",
  "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
  "sec-ch-ua": "\"Chromium\";v=\"143\", \"Not A(Brand\";v=\"24\"",
  "sec-ch-ua-mobile": "?0"
}

POST https://storefrontgateway.supervalu.ie/api/event-tracking/v1/tracking/event
Heade

In [18]:
import requests
import uuid

session = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "x-site-host": "https://shop.supervalu.ie",
    "x-site-location": "HeadersBuilderInterceptor",
    "x-correlation-id": str(uuid.uuid4()),
    "x-shopping-mode": "11111111-1111-1111-1111-111111111111",
    "x-customer-session-id": f"https://shop.supervalu.ie|{uuid.uuid4()}",
    "Referer": "https://shop.supervalu.ie/",
}

store_id = "250"
location_id = "0b556834-14bc-4030-a0be-3f5aca8a9de0"
product_id = "1882507000"

# Try different URL patterns
urls_to_try = [
    f"https://storefrontgateway.supervalu.ie/api/stores/{store_id}/locations/{location_id}/products/{product_id}",
    f"https://storefrontgateway.supervalu.ie/api/stores/{store_id}/products/{product_id}?locationId={location_id}",
    f"https://storefrontgateway.supervalu.ie/api/stores/{store_id}/products/{product_id}",
]

for url in urls_to_try:
    response = session.get(url, headers=headers)
    print(f"{url}")
    print(f"Status: {response.status_code}")
    print(f"Response: {response.text[:200]}\n")

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): storefrontgateway.supervalu.ie:443
DEBUG:urllib3.connectionpool:https://storefrontgateway.supervalu.ie:443 "GET /api/stores/250/locations/0b556834-14bc-4030-a0be-3f5aca8a9de0/products/1882507000 HTTP/1.1" 404 0
DEBUG:urllib3.connectionpool:https://storefrontgateway.supervalu.ie:443 "GET /api/stores/250/products/1882507000?locationId=0b556834-14bc-4030-a0be-3f5aca8a9de0 HTTP/1.1" 200 None


https://storefrontgateway.supervalu.ie/api/stores/250/locations/0b556834-14bc-4030-a0be-3f5aca8a9de0/products/1882507000
Status: 404
Response: 



DEBUG:urllib3.connectionpool:https://storefrontgateway.supervalu.ie:443 "GET /api/stores/250/products/1882507000 HTTP/1.1" 200 None


https://storefrontgateway.supervalu.ie/api/stores/250/products/1882507000?locationId=0b556834-14bc-4030-a0be-3f5aca8a9de0
Status: 200
Response: {"name":"Fonte Real Alicante Bouschet Reserva (75 cl)","defaultCategory":"Portugal","categories":[{"categoryId":"21742f38-b3ec-452c-be59-d153cdb8b74e","retailerId":"Grocery","category":"Grocery","cate

https://storefrontgateway.supervalu.ie/api/stores/250/products/1882507000
Status: 200
Response: {"name":"Fonte Real Alicante Bouschet Reserva (75 cl)","defaultCategory":"Portugal","categories":[{"categoryId":"21742f38-b3ec-452c-be59-d153cdb8b74e","retailerId":"Grocery","category":"Grocery","cate



In [32]:
# Check for a stores endpoint
urls_to_try = [
    "https://storefrontgateway.supervalu.ie/api/stores",
    "https://storefrontgateway.supervalu.ie/api/stores/list",
    "https://shop.supervalu.ie/api/stores",
    "https://storefrontgateway.supervalu.ie/api/v1/stores",
]

for url in urls_to_try:
    response = session.get(url, headers=headers)
    print(f"{url}: {response.status_code}")
    if response.status_code == 200:
        print(response.text[:500])

DEBUG:urllib3.connectionpool:Resetting dropped connection: storefrontgateway.supervalu.ie
DEBUG:urllib3.connectionpool:https://storefrontgateway.supervalu.ie:443 "GET /api/stores HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://storefrontgateway.supervalu.ie:443 "GET /api/stores/list HTTP/1.1" 404 0
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): shop.supervalu.ie:443


https://storefrontgateway.supervalu.ie/api/stores: 200
{"total":143,"items":[{"id":"a266dd83-e1ed-4086-a109-55bca4dc419e","status":"Active","type":"Corporate","name":"SuperValu Online","addressLine1":"Test Site","addressLine2":"Cork","addressLine3":null,"countyProvinceState":"Cork","postCode":"T12 N799","city":"Cork","country":"Ireland","email":null,"phone":"","retailerStoreId":"5550","timeZone":"Europe/Dublin","currency":"EUR","openingHours":null,"location":{"latitude":0.0,"longitude":0.0},"languages":[{"isoCode":"en-IE","isDefault":true}],"siteId"
https://storefrontgateway.supervalu.ie/api/stores/list: 404


DEBUG:urllib3.connectionpool:https://shop.supervalu.ie:443 "GET /api/stores HTTP/1.1" 404 None
DEBUG:urllib3.connectionpool:https://storefrontgateway.supervalu.ie:443 "GET /api/v1/stores HTTP/1.1" 404 0


https://shop.supervalu.ie/api/stores: 404
https://storefrontgateway.supervalu.ie/api/v1/stores: 404


In [34]:
response = session.get("https://storefrontgateway.supervalu.ie/api/stores", headers=headers)

DEBUG:urllib3.connectionpool:https://storefrontgateway.supervalu.ie:443 "GET /api/stores HTTP/1.1" 200 None


In [40]:
json.loads(response.text)['items']

[{'id': 'a266dd83-e1ed-4086-a109-55bca4dc419e',
  'status': 'Active',
  'type': 'Corporate',
  'name': 'SuperValu Online',
  'addressLine1': 'Test Site',
  'addressLine2': 'Cork',
  'addressLine3': None,
  'countyProvinceState': 'Cork',
  'postCode': 'T12 N799',
  'city': 'Cork',
  'country': 'Ireland',
  'email': None,
  'phone': '',
  'retailerStoreId': '5550',
  'timeZone': 'Europe/Dublin',
  'currency': 'EUR',
  'openingHours': None,
  'location': {'latitude': 0.0, 'longitude': 0.0},
  'languages': [{'isoCode': 'en-IE', 'isDefault': True}],
  'siteId': 'f43124d9-814a-4caf-a0fb-3d4f0518e0e5',
  'urls': [],
  'categoryHierarchyId': '3e9310e5-b8a6-485a-9a3f-9495d81e52aa',
  'shoppingModes': ['pickup', 'delivery']},
 {'id': '89f04fa7-1914-472c-9413-74da38ca480b',
  'status': 'Active',
  'type': 'Regular',
  'name': "Moycullen - Kavanagh's",
  'addressLine1': 'An Cearnog Nua',
  'addressLine2': 'Moycullen',
  'addressLine3': None,
  'countyProvinceState': 'Galway',
  'postCode': 'H91 FK

In [111]:
# Filter list of dicts for specific counties
target_counties = ['Cork', 'Dublin', 'Galway']

filtered_stores = [
    {'id': store['id'], 'retailerStoreId': store['retailerStoreId'], 'county': store['countyProvinceState']}
    for store in sv_store_info
    if store.get('countyProvinceState') in target_counties
]

In [125]:
[{s['countyProvinceState']: s['retailerStoreId']} for s in sv_store_info if s['countyProvinceState'] in ['Cork','Dublin']]

[{'Cork': '5550'},
 {'Cork': '250'},
 {'Cork': '260'},
 {'Cork': '262'},
 {'Cork': '267'},
 {'Dublin': '309'},
 {'Cork': '312'},
 {'Cork': '316'},
 {'Cork': '319'},
 {'Dublin': '321'},
 {'Cork': '322'},
 {'Cork': '324'},
 {'Cork': '325'},
 {'Cork': '326'},
 {'Cork': '327'},
 {'Cork': '344'},
 {'Cork': '347'},
 {'Dublin': '350'},
 {'Dublin': '356'},
 {'Dublin': '374'},
 {'Dublin': '382'},
 {'Cork': '491'},
 {'Dublin': '664'},
 {'Cork': '831'},
 {'Dublin': '1625'},
 {'Dublin': '1626'},
 {'Dublin': '1702'},
 {'Dublin': '1703'},
 {'Dublin': '1705'},
 {'Dublin': '1706'},
 {'Dublin': '1708'},
 {'Dublin': '1709'},
 {'Dublin': '1710'},
 {'Dublin': '1712'},
 {'Dublin': '1713'},
 {'Dublin': '1733'},
 {'Dublin': '1741'},
 {'Dublin': '1932'},
 {'Cork': '1991'},
 {'Cork': '1993'},
 {'Dublin': '1714'},
 {'Dublin': '5552'},
 {'Dublin': '5553'},
 {'Dublin': '5551'},
 {'Dublin': '1734'},
 {'Dublin': '5554'},
 {'Dublin': '5556'},
 {'Dublin': '2073'},
 {'Dublin': '5557'},
 {'Dublin': '2327'}]

In [76]:
[(i,j) for i,j in sv_store_info[0].items()]


[('id', 'a266dd83-e1ed-4086-a109-55bca4dc419e'),
 ('status', 'Active'),
 ('type', 'Corporate'),
 ('name', 'SuperValu Online'),
 ('addressLine1', 'Test Site'),
 ('addressLine2', 'Cork'),
 ('addressLine3', None),
 ('countyProvinceState', 'Cork'),
 ('postCode', 'T12 N799'),
 ('city', 'Cork'),
 ('country', 'Ireland'),
 ('email', None),
 ('phone', ''),
 ('retailerStoreId', '5550'),
 ('timeZone', 'Europe/Dublin'),
 ('currency', 'EUR'),
 ('openingHours', None),
 ('location', {'latitude': 0.0, 'longitude': 0.0}),
 ('languages', [{'isoCode': 'en-IE', 'isDefault': True}]),
 ('siteId', 'f43124d9-814a-4caf-a0fb-3d4f0518e0e5'),
 ('urls', []),
 ('categoryHierarchyId', '3e9310e5-b8a6-485a-9a3f-9495d81e52aa'),
 ('shoppingModes', ['pickup', 'delivery'])]

In [ ]:
with open(
    "/Users/brianbarry/Documents/supermarket_scraping/supervalu/supervalu_ids.csv", "w", newline=""
) as f:
    f.write("product_id\n")
    f.write("\n".join(supervalu_ids))

In [126]:
# Get dunnes product

session = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-GB,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Origin": "https://www.dunnesstoresgrocery.com",
    "Referer": "https://www.dunnesstoresgrocery.com/",
    "Connection": "keep-alive",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-site",
}

# First visit the main site to get cookies
session.get("https://www.dunnesstoresgrocery.com/", headers=headers)

# Then try the API
url = "https://storefrontgateway.dunnesstoresgrocery.com/api/stores/258/products/100806905"
response = session.get(url, headers=headers)

print(response.status_code)
product_info = json.loads(response.text)

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): www.dunnesstoresgrocery.com:443
DEBUG:urllib3.connectionpool:https://www.dunnesstoresgrocery.com:443 "GET / HTTP/1.1" 403 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): storefrontgateway.dunnesstoresgrocery.com:443
DEBUG:urllib3.connectionpool:https://storefrontgateway.dunnesstoresgrocery.com:443 "GET /api/stores/258/products/100806905 HTTP/1.1" 200 None


200


In [128]:
product_info

{'name': 'Dunnes Stores Irish Whole Milk 2L',
 'defaultCategory': 'Full Fat Milk',
 'categories': [{'categoryId': '4a320ec5-9f17-0230-e063-0a01002c4618',
   'retailerId': 'Grocery',
   'category': 'Grocery',
   'categoryBreadcrumb': '/Grocery'},
  {'categoryId': '4a320ec5-a066-0230-e063-0a01002c4618',
   'retailerId': '47173',
   'category': 'Chilled Food',
   'categoryBreadcrumb': '/Grocery/Chilled Food'},
  {'categoryId': '4a320ec5-a09c-0230-e063-0a01002c4618',
   'retailerId': '47295',
   'category': 'Fresh Milk & Cream',
   'categoryBreadcrumb': '/Grocery/Chilled Food/Fresh Milk & Cream'},
  {'categoryId': '4a320ec5-a0a1-0230-e063-0a01002c4618',
   'retailerId': '48005',
   'category': 'Full Fat Milk',
   'categoryBreadcrumb': '/Grocery/Chilled Food/Fresh Milk & Cream/Full Fat Milk'}],
 'sku': '100806905',
 'description': "Pasteurised and Homogenised Whole Irish Cow&#x27;s Milk<br/><br/><b>Nutritional Claims</b><br/>Source of calcium<br/>High in protein<br/>High in vitamin B12<br/>